# 02 - Data Cleaning

Minimal cleaning before WoE/IV binning. Only fixes values that would distort bins:

1. `DAYS_EMPLOYED = 365243` placeholder → NaN + flag column `DAYS_EMPLOYED_ANOM`.
2. Meaningless labels (`CODE_GENDER = "XNA"`, `NAME_FAMILY_STATUS = "Unknown"`) → NaN.
3. Negative `DAYS_*` offsets → positive years (readability only; WoE is invariant to monotonic transforms).

**Not done here on purpose:** imputing missing values (WoE keeps them as a separate bin) and dropping columns (decided later from IV).

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add project root to sys.path so `src` can be imported from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.cleaning import DAYS_EMPLOYED_ANOMALY, clean_application
from src.eda import classify_variables

## 2. Load raw data

In [2]:
RAW_PATH = PROJECT_ROOT / "data" / "application_train.csv"
df_raw = pd.read_csv(RAW_PATH)
df_raw.shape

(307511, 122)

## 3. Inspect issues before cleaning

In [3]:
# DAYS_EMPLOYED placeholder: how many clients and how they behave vs. the rest
is_anom = df_raw["DAYS_EMPLOYED"] == DAYS_EMPLOYED_ANOMALY
print(f"Anomalous rows: {is_anom.sum():,} ({is_anom.mean():.2%})")

# Default rate inside vs. outside the anomaly group (is the flag informative?)
df_raw.groupby(is_anom)["TARGET"].mean().rename(index={True: "anomaly", False: "normal"})

Anomalous rows: 55,374 (18.01%)


DAYS_EMPLOYED
normal     0.086600
anomaly    0.053996
Name: TARGET, dtype: float64

In [4]:
# Who are the anomalous clients? (expected: mostly pensioners)
df_raw.loc[is_anom, "NAME_INCOME_TYPE"].value_counts()

NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

In [5]:
# Invalid category labels
print(df_raw["CODE_GENDER"].value_counts(dropna=False), "\n")
print(df_raw["NAME_FAMILY_STATUS"].value_counts(dropna=False))

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64 

NAME_FAMILY_STATUS
Married                 196432
Single / not married     45444
Civil marriage           29775
Separated                19770
Widow                    16088
Unknown                      2
Name: count, dtype: int64


## 4. Apply cleaning

In [6]:
df = clean_application(df_raw)
df.shape

(307511, 123)

## 5. Verify

In [7]:
# Placeholder removed, flag created
assert (df["YEARS_EMPLOYED"] < 0).sum() == 0, "negative years remain"
print(df["DAYS_EMPLOYED_ANOM"].value_counts())

# Invalid labels removed
assert not df["CODE_GENDER"].isin(["XNA"]).any()
assert not df["NAME_FAMILY_STATUS"].isin(["Unknown"]).any()

DAYS_EMPLOYED_ANOM
0    252137
1     55374
Name: count, dtype: int64


In [8]:
# Converted year columns should now be readable (e.g. age 20-70)
year_cols = ["AGE_YEARS", "YEARS_EMPLOYED", "YEARS_REGISTRATION",
             "YEARS_ID_PUBLISH", "YEARS_LAST_PHONE_CHANGE"]
df[year_cols].describe().round(2)

,AGE_YEARS,YEARS_EMPLOYED,YEARS_REGISTRATION,YEARS_ID_PUBLISH,YEARS_LAST_PHONE_CHANGE
count,307511.00,252137.00,307511.00,307511.00,307510.00
mean,43.91,6.53,13.65,8.20,2.64
std,11.95,6.40,9.65,4.13,2.26
min,20.50,0.00,0.00,0.00,0.00
25%,33.98,2.10,5.50,4.71,0.75
50%,43.12,4.51,12.33,8.91,2.07
75%,53.89,8.69,20.48,11.77,4.30
max,69.07,49.04,67.55,19.70,11.75


## 6. Re-classify variables

Column names changed (`DAYS_*` → `YEARS_*`) and `DAYS_EMPLOYED_ANOM` was added, so the lists must be rebuilt.

In [9]:
# Same rules as in 01_eda: count variables forced to numeric
COUNT_VARS = (
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
)
var_types = classify_variables(df, max_categories=10, force_numeric=COUNT_VARS)
var_types["var_type"].value_counts()

var_type
numeric        70
categorical    51
Name: count, dtype: int64

## 7. Save cleaned data

Pickle keeps dtypes and needs no extra dependency. `data/` is git-ignored.

In [10]:
CLEAN_PATH = PROJECT_ROOT / "data" / "application_train_clean.pkl"
df.to_pickle(CLEAN_PATH)
var_types.to_pickle(PROJECT_ROOT / "data" / "var_types.pkl")
print(f"Saved: {CLEAN_PATH}")

Saved: /home/mdtt/Documents/Mdtt/IT/Projects/credit-risk-scorecard/data/application_train_clean.pkl
